
#### Run the cell below to install the required packages for Copilot


In [1]:

#Run this cell to install the required packages for Copilot
%load_ext dscopilot_installer
%activate_dscopilot


StatementMeta(, c2731e78-7056-4cf9-ac81-3f980c288492, 3, Finished, Available, Finished)

# **Code-First Experience**

# Importing the necessary libraries

In [2]:
import os
spark.conf.set("sprk.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")
from pyspark.sql.functions import *
from pyspark.sql.functions import col, unix_timestamp, to_date
from pyspark.sql.types import DateType

StatementMeta(, c2731e78-7056-4cf9-ac81-3f980c288492, 4, Finished, Available, Finished)

In [3]:
df = spark.sql("SELECT * FROM fact_sales LIMIT 100")
display(df)

StatementMeta(, c2731e78-7056-4cf9-ac81-3f980c288492, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3b8ba901-3827-4f5a-a1b9-b409a16ab2b2)

In [4]:
# Code generated by Data Wrangler for PySpark DataFrame

def clean_data(df):
    # Filter rows based on column: 'Fiscal_Quarter'
    df = df.filter(df['Fiscal_Quarter'] == "Q3")
    return df

df_clean = clean_data(df)
display(df_clean)

StatementMeta(, 7c03fe3b-e203-43cd-9722-427800a35e61, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 6cd58eb3-dea7-48dd-b722-d24893cdee41)

# List of tables

In [4]:

TablesList = [
    'dimension_brand.csv',
    'Dimension City.csv',
    'Dimension Customer.csv',
    'dimension_date.csv',
    'dimension_countrydata.csv',
    'dimension_products.csv'
    ]

StatementMeta(, c2731e78-7056-4cf9-ac81-3f980c288492, 6, Finished, Available, Finished)

## Reading the data from source csv

In [ ]:
def LoadFullDataFromSource(Table_Name):
  file_location = "abfss://#wsId#@onelake.dfs.fabric.microsoft.com/#Lakehouse_Bronze#.Lakehouse/Files/Dashboard_KPIs/Dimension_Tables/" + Table_Name
  df = spark.read.format("csv").option("header","true").load(file_location)  
  if "fact_sales" in Table_Name:
      Table_Name = "fact_sales"
      
      print("loading full data for table " + Table_Name + "...")
      df = df.withColumn("TransactionDate",to_date(unix_timestamp(col('TransactionDate'), 'MM-dd-yyyy').cast("timestamp")))
      df = df.withColumn('Year', year(to_date(col("TransactionDate"),"MM-dd-yyyy")))
      df = df.withColumn('Quarter', quarter(to_date(col("TransactionDate"),"MM-dd-yyyy")))
      df = df.withColumn('Month', month(to_date(col("TransactionDate"),"MM-dd-yyyy")))
      df.write.mode("overwrite").format("csv").option("header", "true").partitionBy("Year","Quarter").save("Files/bronze/" + Table_Name)  
      
      print("loaded full data for table " + Table_Name + "!")
  else:
      print("loading full data for table " + Table_Name + "...")
      df.write.mode("overwrite").format("csv").option("header", "true").save("Files/bronze/" + Table_Name)
      print("loaded full data for table " + Table_Name + "!")

StatementMeta(, c2731e78-7056-4cf9-ac81-3f980c288492, 7, Finished, Available, Finished)

# Calling the user defined function

In [ ]:
for table in TablesList:
    LoadFullDataFromSource(table)